# Analisis Hasil TTKI P1
Analisis artefak tersimpan tanpa training. Jalankan dari direktori analysis menggunakan .venv.

In [1]:

from pathlib import Path
import hashlib,json,platform,sys
from datetime import datetime
from zoneinfo import ZoneInfo
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
ANALYSIS=Path.cwd().resolve()
assert ANALYSIS.name=='analysis', 'Jalankan kernel dengan cwd direktori analysis'
RUN=ANALYSIS.parent; ROOT=next(p for p in [RUN,*RUN.parents] if (p/"scripts/reproduce.py").is_file()); ML=RUN/'eksperimen_utama'
def digest(p):
    with p.open('rb') as f:return hashlib.file_digest(f,'sha256').hexdigest()
def read(p):return pd.read_csv(p,float_precision='round_trip')
def write_md(name,lines): (ANALYSIS/name).write_text('\n'.join(lines)+'\n',encoding='utf-8')
def table(frame):
    rows=['| '+' | '.join(map(str,frame.columns))+' |','|'+'|'.join(['---']*len(frame.columns))+'|']
    for record in frame.itertuples(index=False,name=None):
        rows.append('| '+' | '.join(format(v,'.9g') if isinstance(v,float) else str(v) for v in record)+' |')
    return '\n'.join(rows)
validation=[]
for name,h in json.loads((ML/'artifact_checksums.json').read_text()).items():
    assert digest(RUN/name)==h,name
    validation.append({'check':'sha256 '+name,'status':'PASS'})
for name,h in json.loads((RUN/'artifact_checksums.json').read_text()).items():
    assert digest(RUN/name)==h,name
    validation.append({'check':'stage4 sha256 '+name,'status':'PASS'})
M=read(ML/'metrik_utama.csv'); DELTA=read(ML/'selisih_utama.csv'); HIST=read(ML/'historical_comparison_full.csv')
manifest=json.loads((ML/'manifest_eksperimen.json').read_text());params=json.loads((ML/'actual_parameters.json').read_text())
members=read(RUN/'split_membership.csv.gz');splits=read(RUN/'hasil/ringkasan_split.csv');overlap=read(RUN/'hasil/irisan_domain.csv');structure=read(RUN/'hasil/audit_struktur_mentah.csv')
assert len(M)==6 and len(params)==6 and len(list((ML/'predictions').glob('*.csv.gz')))==12
for filename,expected in [('independent_metric_validation.csv',120),('independent_host_validation.csv',138)]:
    v=read(ML/filename);assert len(v)==expected and set(v.status)=={'PASS'}
    validation.append({'check':filename+' all checks PASS','status':'PASS','count':len(v)})
errors=[]
for row in M.to_dict('records'):
    model,scenario=row['model'],row['skenario'];protocol='domain' if scenario=='C' else 'acak'
    assert params[f'{model}/{scenario}']['classes']==[0,1]
    for prefix,part in [('val','validasi'),('uji','uji')]:
        pred=read(ML/'predictions'/f'{model}_{scenario}_{prefix}.csv.gz');expected=members.loc[members[protocol]==part]
        assert np.array_equal(pred.clean_row_id,expected.clean_row_id) and np.array_equal(pred.url_sha256,expected.url_sha256)
        assert np.array_equal(pred.phishing,expected.phishing) and pred.url_sha256.is_unique
        y=pred.phishing.to_numpy();prob=pred.probability_phishing.to_numpy();yp=pred.prediction.to_numpy()
        assert (pred.threshold==row['ambang']).all() and np.array_equal(yp,prob>=row['ambang'])
        tn=int(((y==0)&(yp==0)).sum());fp=int(((y==0)&(yp==1)).sum());fn=int(((y==1)&(yp==0)).sum());tp=int(((y==1)&(yp==1)).sum())
        actual=dict(tn=tn,fp=fp,fn=fn,tp=tp,accuracy=(tp+tn)/len(y),precision=tp/(tp+fp),recall=tp/(tp+fn),f1=2*tp/(2*tp+fp+fn),fpr=fp/(fp+tn),roc_auc=roc_auc_score(y,prob))
        for key,value in actual.items():assert abs(value-row[prefix+'_'+key])<=1e-12,(model,scenario,prefix,key)
        errors.append(dict(model=model,scenario=scenario,partition=part,samples=len(y),legitimate=tn+fp,phishing=tp+fn,tn=tn,fp=fp,fn=fn,tp=tp,total_errors=fp+fn,error_rate=(fp+fn)/len(y),fnr_phishing=fn/(tp+fn),fpr_legitimate=fp/(tn+fp),fn_share_all_samples=fn/len(y),fp_share_all_samples=fp/len(y),fn_share_errors=fn/(fp+fn),fp_share_errors=fp/(fp+fn)))
        validation.append({'check':f'{model}/{scenario}/{part}: predictions membership, counts, threshold and metrics','status':'PASS'})
assert set(overlap.loc[overlap.split=='domain','domain_irisan'])=={0}
for scenario in ['A','B','C']:
    for partition in ['val','uji']:
        p1=read(ML/'predictions'/f'LR_{scenario}_{partition}.csv.gz');p2=read(ML/'predictions'/f'RF_{scenario}_{partition}.csv.gz')
        assert p1.url_sha256.equals(p2.url_sha256)
for model in ['LR','RF']:
    for partition in ['val','uji']:
        assert read(ML/'predictions'/f'{model}_A_{partition}.csv.gz').url_sha256.equals(read(ML/'predictions'/f'{model}_B_{partition}.csv.gz').url_sha256)
historical=read(ROOT/'hasil/metrik_utama.csv')
for row in HIST.to_dict('records'):
    new=M.loc[(M.model==row['model'])&(M.skenario==row['scenario']),row['metric']].iloc[0]
    old=historical.loc[(historical.model==row['model'])&(historical.skenario==row['scenario']),row['metric']].iloc[0]
    assert new==row['reproduction'] and old==row['historical'] and abs(new-old)==row['absolute_difference']
assert HIST.loc[HIST.metric!='uji_batch_ms','absolute_difference'].max()==0
print('Input checksums and 12 prediction files verified; no training executed')


Input checksums and 12 prediction files verified; no training executed


In [2]:

performance=M.copy();performance['feature_count']=performance.skenario.map({'A':38,'B':15,'C':15});performance['protocol']=performance.skenario.map({'A':'random_stratified','B':'random_stratified','C':'domain_grouped'})
metric_names=['accuracy','precision','recall','f1','roc_auc','fpr']
for scenario in ['A','B','C']:
    group=M.loc[M.skenario==scenario].set_index('model')
    for metric in metric_names:
        performance.loc[performance.skenario==scenario,'RF_minus_LR_test_'+metric+'_pp']=100*(group.loc['RF','uji_'+metric]-group.loc['LR','uji_'+metric])
performance.to_csv(ANALYSIS/'performance_comparison.csv',index=False)
scenario_rows=[]
for model in ['LR','RF']:
    group=M.loc[M.model==model].set_index('skenario')
    for later,earlier in [('B','A'),('C','B')]:
        result=dict(model=model,direction=f'{later} minus {earlier}',later=later,earlier=earlier,metric_partition='test',same_test_membership=later=='B',same_feature_set=later=='C')
        for metric in metric_names:result['delta_'+metric+'_pp']=100*(group.loc[later,'uji_'+metric]-group.loc[earlier,'uji_'+metric])
        for count in ['fp','fn','samples']:result['delta_'+count+'_count']=int(group.loc[later,'uji_'+count]-group.loc[earlier,'uji_'+count])
        result['delta_training_seconds']=group.loc[later,'training_seconds']-group.loc[earlier,'training_seconds']
        result['delta_inference_ms_per_1024']=group.loc[later,'uji_batch_ms']-group.loc[earlier,'uji_batch_ms']
        result['training_relative_change_percent']=100*(group.loc[later,'training_seconds']/group.loc[earlier,'training_seconds']-1)
        result['inference_relative_change_percent']=100*(group.loc[later,'uji_batch_ms']/group.loc[earlier,'uji_batch_ms']-1)
        scenario_rows.append(result)
SC=pd.DataFrame(scenario_rows);SC.to_csv(ANALYSIS/'scenario_deltas.csv',index=False)
for row in DELTA.to_dict('records'):
    ours=SC.loc[(SC.model==row['model'])&(SC.direction==row['perbandingan'].replace(' - ',' minus '))].iloc[0]
    for metric in ['f1','recall','fpr']:assert abs(ours['delta_'+metric+'_pp']/100-row['uji_'+metric])<=1e-15
ERR=pd.DataFrame(errors);ERR.to_csv(ANALYSIS/'error_analysis.csv',index=False)
efficiency=M[['model','skenario','training_seconds','uji_batch_ms','batch_size']].copy()
for scenario in ['A','B','C']:
    g=M.loc[M.skenario==scenario].set_index('model');mask=efficiency.skenario==scenario
    efficiency.loc[mask,'RF_minus_LR_training_seconds']=g.loc['RF','training_seconds']-g.loc['LR','training_seconds']
    efficiency.loc[mask,'RF_over_LR_training_ratio']=g.loc['RF','training_seconds']/g.loc['LR','training_seconds']
    efficiency.loc[mask,'RF_minus_LR_inference_ms']=g.loc['RF','uji_batch_ms']-g.loc['LR','uji_batch_ms']
    efficiency.loc[mask,'RF_over_LR_inference_ratio']=g.loc['RF','uji_batch_ms']/g.loc['LR','uji_batch_ms']
efficiency['measurement']='pipeline.fit seconds; inference median5, warmup1, batch1024, thread1; excludes URL extraction and page retrieval'
efficiency.to_csv(ANALYSIS/'efficiency_analysis.csv',index=False)
TEST=M[['model','skenario','uji_accuracy','uji_precision','uji_recall','uji_f1','uji_roc_auc','uji_fpr']]
RF_COMPARE=performance.loc[performance.model=='RF',['skenario']+['RF_minus_LR_test_'+m+'_pp' for m in metric_names]]
print(table(SC[['model','direction','delta_f1_pp','delta_recall_pp','delta_precision_pp','delta_fpr_pp','delta_fp_count','delta_fn_count']]))


| model | direction | delta_f1_pp | delta_recall_pp | delta_precision_pp | delta_fpr_pp | delta_fp_count | delta_fn_count |
|---|---|---|---|---|---|---|---|
| LR | B minus A | -0.63133706 | -1.27337843 | 0.0192907422 | -0.0148309274 | -3 | 192 |
| LR | C minus B | -0.160036032 | -0.29845818 | -0.0175012632 | 0.0142107097 | 3 | 56 |
| RF | B minus A | -0.45273963 | -0.802493699 | -0.100227182 | 0.0741546371 | 15 | 121 |
| RF | C minus B | -0.0987154852 | -0.231138021 | 0.0360629211 | -0.0253384302 | -5 | 42 |


In [3]:

limitations=[
('Struktur kelas tidak seimbang','Seluruh 134850 legitimate mentah adalah akar; path/query hanya phishing. Panjang path/query dapat menjadi shortcut korpus. Ini potensi shortcut learning, bukan bukti mekanisme yang dipelajari setiap model atau leakage terkonfirmasi.','../hasil/audit_struktur_mentah.csv baris 2-4'),
('Satu dataset','Domain baru dalam PhiUSIIL tidak membuktikan generalisasi lintas sumber, waktu atau deployment. FPR hanya mewakili jenis legitimate yang tersedia.','../hasil/ringkasan_split.csv; ../hasil/audit_struktur_mentah.csv'),
('Satu seed','Seed42 menghasilkan satu realisasi split/model. Tidak ada pengulangan seed, interval ketidakpastian atau pengujian signifikansi pada tahap ini. Selisih dilaporkan deskriptif.','../eksperimen_utama/manifest_eksperimen.json seed'),
('Baseline selected features','A memakai 15 leksikal+23 atribut halaman terpilih, bukan seluruh fitur sumber atau seluruh fitur relevan yang terbukti lengkap. Hasil A/B terbatas pada penambahan 23 atribut tersebut.','../../../docs/audit/FEATURE_AUDIT.md; ../eksperimen_utama/actual_parameters.json'),
('Komposisi B/C berbeda','Test B=35306, phishing15078, legitimate20228; C=36229, phishing15744, legitimate20485. Membership/protokol berbeda; jangan mengatribusikan semua perubahan kepada domain secara kausal atau membandingkan counts tanpa denominator.','../hasil/ringkasan_split.csv baris 4 dan 7; ../split_membership.csv.gz'),
('Provenance belum lengkap','Versi/tanggal unduh/lisensi salinan Kaggle dan definisi upstream beberapa fitur belum lengkap. Skor statistik yang dikeluarkan tidak otomatis menyebabkan leakage; eksklusi konservatif dan cakupan eksperimen harus dibedakan.','../../../docs/audit/feature_registry.csv; ../../../docs/audit/FEATURE_AUDIT.md'),
('Pengukuran waktu','Satu run, satu perangkat, median5 pada batch1024. Train pipeline.fit; inferensi mencakup preprocessing/predict_proba tetapi mengecualikan ekstraksi URL dan pengambilan halaman. Bukan latensi end-to-end atau benchmark stabilitas. Tidak ada distribusi lima waktu mentah tersimpan.','../eksperimen_utama/manifest_eksperimen.json; ../eksperimen_utama/metrik_utama.csv'),
('Kesamaan historis terbatas','Threshold, confusion counts dan metrik agregat cocok tepat pada CSV. Prediksi/membership historis per record tidak tersedia; tidak dapat mengklaim kesamaan per record atau reproducibility sempurna. Python/perangkat berbeda.','../eksperimen_utama/historical_comparison_full.csv; ../EXPERIMENT_REPORT.md'),
('Persetujuan baseline','Otorisasi training pengguna hanya untuk reproduksi historis. Persetujuan final baseline seluruh tim masih menunggu.','../../../docs/audit/BASELINE_DECISION.md; ../stage5_execution/authorization.json'),
('Validation versus test','Threshold memaksimalkan F1 validation; F1 validation pada threshold terpilih merupakan hasil seleksi pada validation, bukan estimasi independen. Test tidak dipakai memilih threshold, fitur atau parameter dalam runner.','../02_Eksperimen_Utama.executed.ipynb cell 4-5'),
('Tidak ada hasil subgroup baru','Tahap6 tidak menambah subgroup/reweighting. Counts struktur mentah digunakan hanya untuk membatasi interpretasi dataset, bukan mengevaluasi subgroup model baru.','analysis_results.ipynb; ../hasil/audit_struktur_mentah.csv')]
write_md('LIMITATIONS.md',['# Keterbatasan dan Batas Klaim','','Tahap6 menganalisis artefak tersimpan tanpa training/tuning atau perubahan fitur/split.']+sum(([f'## {title}','',text,'','Bukti: '+evidence+'.',''] for title,text,evidence in limitations),[])+['Tidak ada dasar untuk menyebut perbedaan signifikan secara statistik. Tidak ada bukti baru yang mengonfirmasi data leakage. Tidak mengklaim model terbaik untuk seluruh kondisi operasional.'])
rq=[]
for model in ['LR','RF']:
    ab=SC.loc[(SC.model==model)&(SC.later=='B')].iloc[0];bc=SC.loc[(SC.model==model)&(SC.later=='C')].iloc[0]
    rq.append({'Pertanyaan penelitian':f'{model}: bagaimana pembatasan fitur mengubah F1/recall/FPR?','Bukti eksperimen':'B minus A, membership acak identik','Nilai metrik':f"F1 {ab.delta_f1_pp:+.6f} pp; recall {ab.delta_recall_pp:+.6f} pp; FPR {ab.delta_fpr_pp:+.6f} pp",'Interpretasi':'F1 dan recall turun ketika 23 atribut halaman terpilih dikeluarkan; arah FPR bergantung model.','Keterbatasan':'Selected features; satu dataset/seed; tidak ada uji signifikansi.','Lokasi artefak pendukung':'scenario_deltas.csv; ../eksperimen_utama/metrik_utama.csv; ../split_membership.csv.gz'})
    rq.append({'Pertanyaan penelitian':f'{model}: bagaimana protokol domain mengubah F1/recall/FPR dengan fitur identik?','Bukti eksperimen':'C minus B; 15 fitur identik, C nol irisan domain','Nilai metrik':f"F1 {bc.delta_f1_pp:+.6f} pp; recall {bc.delta_recall_pp:+.6f} pp; FPR {bc.delta_fpr_pp:+.6f} pp",'Interpretasi':'F1 dan recall turun pada realisasi split C; FPR LR naik, RF turun.','Keterbatasan':'Test dan prevalensi berubah; bukan efek kausal domain yang terisolasi.','Lokasi artefak pendukung':'scenario_deltas.csv; ../hasil/ringkasan_split.csv; ../hasil/irisan_domain.csv'})
RQ=pd.DataFrame(rq)
write_md('RESEARCH_QUESTION_ANSWERS.md',['# Jawaban Research Question Berdasarkan Bukti','','RQ tetap: bagaimana F1 kelas phishing, recall dan FPR LR/RF berubah ketika fitur dibatasi menjadi leksikal URL dan test dipisahkan berdasarkan domain terdaftar?','','Semua selisih dihitung presisi penuh; pp berarti poin persentase, yaitu 100 x selisih proporsi. Pembulatan hanya penyajian.','',table(RQ),'','Kesimpulan deskriptif: fitur halaman terpilih memberi skor F1/recall lebih tinggi pada split A/B yang sama. Dengan fitur leksikal tetap, split C menunjukkan F1/recall lebih rendah dari B pada run ini. FPR tidak bergerak seragam. Tidak ada klaim kausal mutlak atau signifikansi statistik.','', 'Status baseline tim tetap menunggu persetujuan; otorisasi reproduksi tidak menggantikannya.'])
claims=[
('RF mempunyai F1/recall lebih tinggi dari LR di A/B/C','performance_comparison.csv; ../eksperimen_utama/metrik_utama.csv','RF_minus_LR_test_f1_pp dan recall_pp','Deskriptif pada enam hasil ini; RF B memiliki FPR lebih tinggi dari LR B.'),
('Pembatasan fitur B minus A menurunkan F1/recall kedua model','scenario_deltas.csv; ../split_membership.csv.gz','Dua baris B minus A; membership prediksi A/B diperiksa sama','Tidak berarti semua fitur halaman sumber diperlukan; hanya 23 atribut terpilih dibandingkan.'),
('C minus B menurunkan F1/recall pada run ini','scenario_deltas.csv; ../hasil/ringkasan_split.csv','Dua baris C minus B','Membership/prevalensi berubah; bukan kausalitas domain mutlak.'),
('C memiliki nol irisan domain antarsplit','../hasil/irisan_domain.csv; ../split_membership.csv.gz','Baris domain: ketiga pasangan 0','Definisi PSL private dan fallback sesuai Tahap4; bukan generalisasi lintas korpus.'),
('FPR LR turun A->B lalu naik B->C; RF sebaliknya','scenario_deltas.csv; error_analysis.csv','delta_fpr_pp, fp/legitimate','Jumlah FP kecil; tidak disertai ketidakpastian; zero FP RF/A bukan jaminan FPR populasi nol.'),
('RF/A menghasilkan kesalahan test paling sedikit','error_analysis.csv','Test RF/A: FP0, FN6, N35306','Bukan bukti performa populasi sempurna atau keunggulan signifikan.'),
('LR lebih cepat pada training dan inferensi di tiap skenario','efficiency_analysis.csv','RF_over_LR ratios >1','Hanya satu pengukuran run; waktu bukan end-to-end.'),
('Agregat dan threshold sama dengan historis, timing berbeda','../eksperimen_utama/historical_comparison_full.csv','Seluruh non-timing absolute_difference=0','Tidak membuktikan kesamaan prediksi historis per record.'),
('Semua legitimate mentah pada struktur akar','../hasil/audit_struktur_mentah.csv','Akar134850, path0, query0 legitimate','Potensi shortcut dataset, bukan leakage terkonfirmasi.'),
('Tidak ada klaim signifikansi statistik','LIMITATIONS.md','Tidak ada uji signifikansi/interval/pengulangan seed','Perbedaan hanya deskriptif.'),
('Integritas input dan hasil lama terjaga','validation_manifest.json','Checksum inventory dan snapshot before/after','Tidak ada training atau modifikasi metrik sumber.'),
('Keputusan baseline belum disetujui seluruh tim','../../../docs/audit/BASELINE_DECISION.md; ../stage5_execution/authorization.json','Status pending; otorisasi training terpisah','Dokumen analisis tidak menetapkan metode baru.')]
write_md('CLAIM_EVIDENCE_MATRIX.md',['# Matriks Klaim dan Bukti','',table(pd.DataFrame(claims,columns=['Klaim','Lokasi bukti','Nilai/field pendukung','Batas klaim'])),'','Tabel jawaban RQ yang memuat nilai aktual tersedia di RESEARCH_QUESTION_ANSWERS.md. Semua path relatif terhadap direktori analysis.'])


In [4]:

err_test=ERR.loc[ERR.partition=='uji'].copy()
err_test['FNR_percent']=100*err_test.fnr_phishing;err_test['FPR_percent']=100*err_test.fpr_legitimate
composition=splits.loc[splits.bagian=='uji'].copy();composition['phishing_prevalence_percent']=100*composition.phishing/composition.baris
notes=[
'# Analisis Ilmiah Hasil Eksperimen TTKI P1','',
'Run reproduction-001; commit d80117b50f179cfefe91f1c138706264a2934cc7. Analisis artefak tersimpan, tanpa training/tuning, perubahan fitur/split, subgroup baru atau reweighting. Bukan draf artikel jurnal. Persetujuan final baseline tim masih menunggu; pengguna telah mengotorisasi reproduksi historis.','',
'## Validasi input dan satuan','',
'Checksum Tahap4 dan seluruh input/output pada inventory Tahap5 diverifikasi. Dua belas file prediksi diperiksa terhadap membership, label, threshold, confusion counts dan seluruh metrik, termasuk ROC-AUC. Validasi Tahap5 120 pemeriksaan notebook dan 138 pemeriksaan host seluruhnya PASS; Tahap6 membaca dan memeriksanya kembali. Bukti rinci disimpan di validation_manifest.json. Seluruh perhitungan memakai float64 dari CSV dengan parser round_trip; pembulatan hanya tabel. pp = poin persentase = 100 x (proporsi kemudian - proporsi awal), bukan perubahan relatif persen. CSV count dan satuan timing eksplisit.','',
'## Performa enam konfigurasi','',table(TEST),'',
'Accuracy, precision, recall, F1, ROC-AUC dan FPR memakai skala proporsi 0-1. performance_comparison.csv menyimpan seluruh metrik validation/test, confusion counts, threshold, ukuran sampel dan feature count. Phishing=1. FPR=FP/(FP+TN).','',
'Perbandingan RF minus LR pada skenario yang sama (pp):','',table(RF_COMPARE),'',
'RF memberi F1/recall/ROC-AUC lebih tinggi pada ketiga skenario dan accuracy lebih tinggi. RF A precision lebih tinggi dan FPR lebih rendah; RF B precision lebih rendah dan FPR lebih tinggi daripada LR B; RF C FPR sama. Pemilihan model melibatkan trade-off recall/FP/biaya, bukan satu pemenang universal. Tidak ada pengujian signifikansi.','',
'## B dikurangi A: pembatasan fitur','',table(SC.loc[SC.later=='B']),'',
'A/B menggunakan train, validation dan test identik, diverifikasi dari ID/hash prediksi dan membership. B mengeluarkan 23 atribut halaman terpilih dari A sambil mempertahankan 15 fitur leksikal. F1 dan recall turun pada LR maupun RF. LR mengalami FN bertambah192 dan FP berkurang3; RF mengalami FN bertambah121 dan FP bertambah15. Precision LR sedikit naik tetapi recall turun; precision RF turun. FPR tidak bergerak seragam. Ini bukti kontribusi 23 atribut terpilih dalam konfigurasi ini, bukan klaim tentang semua fitur sumber.','',
'## C dikurangi B: perubahan protokol split','',table(SC.loc[SC.later=='C']),'',
'B/C memakai 15 fitur identik. Namun train/validation/test dan komposisi kelas berubah. F1/recall menurun pada C untuk kedua model. LR FP naik3 dan FN naik56; RF FP turun5 dan FN naik42. Counts harus dibaca bersama denominator; FN C lebih banyak tidak boleh ditafsirkan tanpa FNR dan jumlah phishing. FPR LR naik, RF turun. Tidak dapat menyatakan pemisahan domain menyebabkan seluruh selisih secara kausal.','',table(composition),'',
'## Kesalahan klasifikasi dan denominator','',table(err_test[['model','scenario','samples','legitimate','phishing','tn','fp','fn','tp','total_errors','error_rate','FNR_percent','FPR_percent']]),'',
'error_analysis.csv berisi validation dan test. FNR=FN/jumlah phishing; FPR=FP/jumlah legitimate; error_rate=(FP+FN)/N. Kolom fn_share_all_samples/fp_share_all_samples memakai denominator N, sedangkan fn_share_errors/fp_share_errors memakai total error. Ini proporsi berbeda dan tidak saling dipertukarkan. Error didominasi FN. RF/A memiliki 6 FN dan 0 FP pada test, tetapi tidak membuktikan FPR populasi nol. B/C harus dibandingkan melalui rate serta denominator, bukan counts mentah saja.','',
'## Efisiensi komputasi','',table(efficiency),'',
'LR lebih cepat daripada RF pada setiap skenario untuk pipeline.fit dan batch inference. A lebih lambat daripada B untuk kedua model, disertai F1/recall lebih tinggi. RF mempertahankan F1/recall lebih tinggi dengan biaya komputasi lebih besar. Perubahan B/C tidak hanya terkait metode split: ukuran dan komposisi train berbeda. Semua waktu berasal dari satu run Tahap5, tidak diukur ulang. Inference median5/warmup1/thread1 pada batch1024; preprocessing pipeline dan predict_proba masuk, ekstraksi URL/pengambilan halaman tidak masuk. Jangan menyebut timing end-to-end atau keunggulan latency stabil di semua perangkat.','',
'## Reproduksibilitas','',
'Seluruh threshold, confusion counts dan metrik validation/test yang tersedia pada historical_comparison_full.csv memiliki selisih tepat nol dengan CSV historis. Waktu inference berbeda, selisih absolut maksimum '+format(HIST.loc[HIST.metric=='uji_batch_ms','absolute_difference'].max(),'.9g')+' ms/batch. Training time historis tidak tersedia. Prediksi historis per record tidak tersedia; kesamaan statistik/threshold tidak membuktikan keanggotaan/prediksi historis identik. Python3.13.0 berbeda dari historis3.14.4; CPU/OS berbeda, lima versi paket utama tercatat sama. Tidak mengklaim reproducibility sempurna.','',
'## Jawaban research question','',table(RQ),'',
'## Keterbatasan dan batas klaim','',
'Lihat LIMITATIONS.md untuk ketimpangan struktur kelas, shortcut potensial, satu dataset/seed, selected features, perubahan komposisi B/C, provenance yang belum lengkap, timing, kesamaan historis terbatas dan status baseline. Tidak ada bukti baru leakage terkonfirmasi dan tidak ada dasar klaim signifikansi statistik.','',
'## Integritas, artefak dan pengulangan analisis','',
'Seluruh output tahap ini berada di analysis/. Notebook analysis_results.ipynb menyimpan source dan hasil eksekusi, dapat dijalankan dengan cwd analysis dan interpreter .venv tanpa training ulang. Ia membaca metrik/prediksi lama dan menulis ulang hanya output analysis. Root notebook, dataset, hasil historis dan hasil eksperimen tidak diubah. Snapshot checksum sebelum/sesudah dicatat pada validation_manifest.json oleh orkestrator.','',
'File baru: ANALYSIS_REPORT.md, RESEARCH_QUESTION_ANSWERS.md, LIMITATIONS.md, performance_comparison.csv, scenario_deltas.csv, error_analysis.csv, efficiency_analysis.csv, CLAIM_EVIDENCE_MATRIX.md, analysis_results.ipynb, serta validation_manifest.json sebagai bukti validasi/integritas. Tidak commit/push.','',
'## Rekomendasi visualisasi tahap berikutnya','',
'1. Plot titik F1/recall/FPR per model dan A/B/C, dengan sumbu FPR terpisah agar skala kecil terbaca. Hindari error bar karena ketidakpastian belum diestimasi.',
'2. Plot delta B-A dan C-B dalam pp, pisahkan panel dan beri catatan test B/C berbeda; bukan diagram efek kausal.',
'3. Confusion matrix dengan count serta normalisasi menurut kelas aktual. Tampilkan N legitimate/phishing pada tiap panel.',
'4. Plot F1 versus batch inference time dengan label konfigurasi; nyatakan satu run, non-end-to-end, tanpa klaim benchmark perangkat universal.',
'5. Diagram komposisi test B/C dan irisan domain historis sebagai konteks interpretasi. Tidak menambah subgroup evaluasi model.','',
'Satu pekerjaan berikutnya: review hasil dan batas klaim bersama tim untuk menyepakati baseline selected features dan bentuk visualisasi pelaporan; tidak perlu training baru hanya untuk mengejar angka historis.'
]
write_md('ANALYSIS_REPORT.md',notes)
record={'analysis_status':'PASS','created_at':datetime.now(ZoneInfo('Asia/Bangkok')).isoformat(),'training_executed':False,'checks':validation,'predictions_files_checked':12,'historical_non_timing_max_absolute_difference':float(HIST.loc[HIST.metric!='uji_batch_ms','absolute_difference'].max()),'baseline_team_approval':'pending','python':platform.python_version(),'input_checksums':{p.relative_to(RUN).as_posix():digest(p) for p in [RUN/'EXPERIMENT_REPORT.md',ML/'metrik_utama.csv',ML/'selisih_utama.csv',ML/'historical_comparison_full.csv',ML/'independent_metric_validation.csv',ML/'independent_host_validation.csv',ML/'actual_parameters.json',ML/'manifest_eksperimen.json',RUN/'split_membership.csv.gz',RUN/'hasil/ringkasan_split.csv',RUN/'hasil/irisan_domain.csv',RUN/'hasil/audit_struktur_mentah.csv']}}
existing=ANALYSIS/'validation_manifest.json'
if existing.exists():
    old=json.loads(existing.read_text());record['previous_integrity_evidence']=old.get('integrity',{})
existing.write_text(json.dumps(record,indent=2),encoding='utf-8')
display(TEST)
display(SC)
display(err_test)
print('Analysis completed; source experiment artifacts unchanged; no training executed')


,model,skenario,uji_accuracy,uji_precision,uji_recall,uji_f1,uji_roc_auc,uji_fpr
0,LR,A,0.999377,0.999337,0.999204,0.999270,0.999975,0.000494
1,LR,B,0.994024,0.999530,0.986470,0.992957,0.996919,0.000346
2,LR,C,0.992547,0.999355,0.983486,0.991357,0.996278,0.000488
3,RF,A,0.999830,1.000000,0.999602,0.999801,0.999998,0.000000
4,RF,B,0.995978,0.998998,0.991577,0.995274,0.998093,0.000742
5,RF,C,0.995059,0.999358,0.989266,0.994286,0.997540,0.000488


,model,direction,later,earlier,metric_partition,same_test_membership,same_feature_set,delta_accuracy_pp,delta_precision_pp,delta_recall_pp,delta_f1_pp,delta_roc_auc_pp,delta_fpr_pp,delta_fp_count,delta_fn_count,delta_samples_count,delta_training_seconds,delta_inference_ms_per_1024,training_relative_change_percent,inference_relative_change_percent
0,LR,B minus A,B,A,test,True,False,-0.535320,0.019291,-1.273378,-0.631337,-0.305505,-0.014831,-3,192,0,-1.545716,-1.3583,-64.507162,-33.091334
1,LR,C minus B,C,B,test,False,True,-0.147627,-0.017501,-0.298458,-0.160036,-0.064133,0.014211,3,56,923,0.257790,-0.2129,30.311229,-7.751966
2,RF,B minus A,B,A,test,True,False,-0.385204,-0.100227,-0.802494,-0.452740,-0.190588,0.074155,15,121,0,-11.285004,-0.8679,-65.285951,-8.003578
3,RF,C minus B,C,B,test,False,True,-0.091881,0.036063,-0.231138,-0.098715,-0.055227,-0.025338,-5,42,923,-0.138007,-0.5464,-2.299923,-5.477145


,model,scenario,partition,samples,legitimate,phishing,tn,fp,fn,tp,total_errors,error_rate,fnr_phishing,fpr_legitimate,fn_share_all_samples,fp_share_all_samples,fn_share_errors,fp_share_errors,FNR_percent,FPR_percent
1,LR,A,uji,35306,20228,15078,20218,10,12,15066,22,0.000623,0.000796,0.000494,0.000340,0.000283,0.545455,0.454545,0.079586,0.049436
3,LR,B,uji,35306,20228,15078,20221,7,204,14874,211,0.005976,0.013530,0.000346,0.005778,0.000198,0.966825,0.033175,1.352965,0.034605
5,LR,C,uji,36229,20485,15744,20475,10,260,15484,270,0.007453,0.016514,0.000488,0.007177,0.000276,0.962963,0.037037,1.651423,0.048816
7,RF,A,uji,35306,20228,15078,20228,0,6,15072,6,0.000170,0.000398,0.000000,0.000170,0.000000,1.000000,0.000000,0.039793,0.000000
9,RF,B,uji,35306,20228,15078,20213,15,127,14951,142,0.004022,0.008423,0.000742,0.003597,0.000425,0.894366,0.105634,0.842287,0.074155
11,RF,C,uji,36229,20485,15744,20475,10,169,15575,179,0.004941,0.010734,0.000488,0.004665,0.000276,0.944134,0.055866,1.073425,0.048816


Analysis completed; source experiment artifacts unchanged; no training executed
